# Wavebit Tutorial 2
## Gradient and Optimization with [JAX](https://jax.readthedocs.io) and [Optax](https://optax.readthedocs.io/)

In [ ]:

import optax

import jax.numpy as jnp

import wavebit

In [ ]:
backend = 1  # JAX

# common
psi = jnp.pi / 2  # pure s (TE-like); psi is the linear angle

top = 1.0  # superstrate: relative permittivity. 1.0 is vacuum
bot = 1.0  # substrate

theta = 0 * jnp.pi / 180  # angle of incidence
phi = 0 * jnp.pi / 180  # angle of rotation

wavelength = 900

thickness = [1000., 500.]  # thickness of each layer, from bottom to top.
period = [1000.]  # length of the unit cell. Here it's 1D.

truncation_order = [10]

complex_dtype = jnp.complex128

In [ ]:
structure_1d_m = jnp.array([
    [
        [1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
    ],
    [
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0],
    ],
]) * 24. + 1.  # relative permittivity: 1 (vacuum) and 25

# The design variable is the permittivity itself, one number per pixel. That is what a
# raster is for -- it is a sampled field rather than an assignment, so every pixel is a
# knob an optimizer can turn. A gradient taken here is d(loss)/d(eps); it was
# d(loss)/d(n) before the material convention changed, and the two differ by the chain
# factor 2n, so numbers from an older run of this notebook do not carry over.

## 2.1 Gradient

Gradient can be calculated with the help of `jax.value_and_grad` function.
Read this for further information: [Automatic Differentiation](https://jax.readthedocs.io/en/latest/jax-101/04-advanced-autodiff.html)

Optax is used for optimization. Like PyTorch, Optax also provides various loss functions and optimizers so users easily can utilize well-established implementations. Refer this tutorial: [Learn Optax](https://optax.readthedocs.io/en/latest/optax-101.html)


Gradient can be utilized to solve optimization problems. Here are examples that show couple of ways to get gradient or optimized values with or without predefined functions of wavebit.

### 2.1.1 Examples

In [ ]:
class Loss:
    def __call__(self, result, *args, **kwargs):
        given, res_te, res_ti = result.incident, result.s_inc, result.p_inc
        T = given.T
        center = [a // 2 for a in T.shape]
        res = T[center[0], center[1]+1]

        return res


loss_fn = Loss()

In [ ]:
solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi, truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure_1d_m, thickness=thickness, complex_dtype=complex_dtype)

parameters = ['structure', 'thickness']
# A single eager gradient can derive the current geometry directly.
forward = solver.solve

# case 1: Gradient
grad = solver.grad(parameters, forward, loss_fn)

print('structure gradient:')
print(grad['structure'])
print('thickness gradient:')
print(grad['thickness'])

## 2.2 Optimization

### 2.2.1 Examples

Example 1

In [ ]:
solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi, truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure_1d_m, thickness=thickness, complex_dtype=complex_dtype)

parameters = ['structure', 'thickness']
# fit prepares the current geometry outside JIT before every optimizer step.
# The solve and gradient remain compiled, including when the pattern changes.
forward = solver.solve

# case 2: SGD
optimizer = optax.sgd(learning_rate=1e-2, momentum=0.9)
res = solver.fit(parameters, forward, loss_fn, optimizer, iteration=3)

print('structure final:')
print(res['structure'])
print('thickness final:')
print(res['thickness'])